# Payments_Reconciliation_Pipeline

## Setup imports and data read paths

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    TimestampType
)

from delta.tables import DeltaTable
from datetime import datetime

# Base location in Databricks File System
BASE_PATH = "/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation"

RAW_PATH = f"{BASE_PATH}/data"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"
DQ_PATH = f"{BASE_PATH}/data_quality"

print("Pipeline initialized")

Pipeline initialized


## Read transaction dataframe

In [0]:
transactions_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation/data/transactions.csv")
)

display(transactions_df)

txn_id,amount,timestamp,status,merchant_id,currency,channel
TXN001,1250.0,2026-07-05T03:45:22.000Z,success,Merch_101,INR,upi
TXN002,3499.5,2026-07-05T03:48:45.000Z,success,Merch_102,INR,card
TXN003,750.0,2026-07-05T03:52:10.000Z,failed,Merch_101,INR,upi
TXN004,12000.0,2026-07-05T04:35:33.000Z,success,Merch_103,INR,imps
TXN005,299.0,2026-07-05T04:42:58.000Z,success,Merch_104,INR,upi
TXN006,5600.75,2026-07-05T05:15:19.000Z,pending,Merch_102,INR,card
TXN007,1890.25,2026-07-05T05:32:44.000Z,success,Merch_105,INR,upi
TXN008,999.0,2026-07-05T05:45:07.000Z,success,Merch_101,INR,upi
TXN009,4500.0,2026-07-05T06:00:55.000Z,failed,Merch_103,INR,imps
TXN010,15999.0,2026-07-05T06:35:22.000Z,success,Merch_106,INR,card


## Read settlements dataframe

In [0]:
settlements_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation/data/settlements.csv")
    .withColumn(
        "settlement_date",
        F.to_date("settlement_date")
    )
)

display(settlements_df)

txn_id,settled_amount,settlement_date,bank_ref,settlement_status,bank_name
TXN001,1249.0,2026-07-06,BNKREF1001,settled,HDFC
TXN002,3498.5,2026-07-06,BNKREF1002,settled,ICICI
TXN004,11995.0,2026-07-06,BNKREF1004,settled,AXIS
TXN005,299.0,2026-07-06,BNKREF1005,settled,HDFC
TXN006,5599.75,2026-07-06,BNKREF1006,settled,ICICI
TXN007,1889.25,2026-07-06,BNKREF1007,settled,SBIN
TXN008,999.0,2026-07-06,BNKREF1008,settled,HDFC
TXN010,15990.0,2026-07-06,BNKREF1010,settled,ICICI
TXN011,899.5,2026-07-06,BNKREF1011,settled,AXIS
TXN012,2248.0,2026-07-06,BNKREF1012,settled,ICICI


## Read UPI response JSON

In [0]:
upi_df = (
    spark.read
    .option("multiLine", True)
    .option("inferSchema", True)
    .json("/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation/data/upi_responses.json")
    .withColumn(
        "timestamp",
        F.to_timestamp("timestamp")
    )
)

display(upi_df)

error_message,note,payee_vpa,payer_vpa,response_code,timestamp,txn_id,upi_ref
null,null,merch101@upi,user1@paytm,00,2026-07-05T03:45:24.000Z,TXN001,UPIREF100001
null,"card transaction, UPI log not applicable",null,null,00,2026-07-05T03:48:47.000Z,TXN002,null
Insufficient funds,null,merch101@upi,user2@phonepe,05,2026-07-05T03:52:12.000Z,TXN003,null
null,null,merch104@upi,user3@gpay,00,2026-07-05T04:43:01.000Z,TXN005,UPIREF100005
null,null,merch105@upi,user4@paytm,00,2026-07-05T05:32:46.000Z,TXN007,UPIREF100007
Timeout at issuer,null,merch101@upi,user5@phonepe,99,2026-07-05T05:45:09.000Z,TXN008,null
Insufficient funds,null,merch103@upi,user6@gpay,05,2026-07-05T06:00:57.000Z,TXN009,null
null,null,merch104@upi,user7@paytm,00,2026-07-05T06:48:35.000Z,TXN011,UPIREF100011
null,null,merch107@upi,user8@phonepe,00,2026-07-05T07:32:50.000Z,TXN013,UPIREF100013
null,null,merch101@upi,user9@gpay,00,2026-07-05T08:25:44.000Z,TXN015,UPIREF100015


## Write Bronze Delta tables

In [0]:
ingestion_ts = datetime.utcnow()

transactions_bronze = (
    transactions_df
    .withColumn("MESSAGE_ARRIVAL_TIMESTAMP", F.current_timestamp())
    .withColumn("source_system", F.lit("internal_ledger"))
)

settlements_bronze = (
    settlements_df
    .withColumn("MESSAGE_ARRIVAL_TIMESTAMP", F.current_timestamp())
    .withColumn("source_system", F.lit("bank"))
)

upi_bronze = (
    upi_df
    .withColumn("MESSAGE_ARRIVAL_TIMESTAMP", F.current_timestamp())
    .withColumn("source_system", F.lit("upi_npci"))
)

transactions_bronze.write.format("delta").mode("overwrite").save(
    f"{BASE_PATH}/bronze/transactions"
)
settlements_bronze.write.format("delta").mode("overwrite").save(
    f"{BASE_PATH}/bronze/settlements"
)
upi_bronze.write.format("delta").mode("overwrite").save(
    f"{BASE_PATH}/bronze/upi_responses"
)

print("Bronze layer created")

/home/spark-7118c2ec-2add-4b4c-ae05-6b/.ipykernel/77/command-8905209690843849-1687430087:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingestion_ts = datetime.utcnow()


Bronze layer created


## Read Bronze

In [0]:
transactions_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze/transactions")
settlements_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze/settlements")
upi_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze/upi_responses")

display(settlements_bronze)

txn_id,settled_amount,settlement_date,bank_ref,settlement_status,bank_name,MESSAGE_ARRIVAL_TIMESTAMP,source_system
TXN001,1249.0,2026-07-06,BNKREF1001,settled,HDFC,2026-08-13T13:21:39.350Z,bank
TXN002,3498.5,2026-07-06,BNKREF1002,settled,ICICI,2026-08-13T13:21:39.350Z,bank
TXN004,11995.0,2026-07-06,BNKREF1004,settled,AXIS,2026-08-13T13:21:39.350Z,bank
TXN005,299.0,2026-07-06,BNKREF1005,settled,HDFC,2026-08-13T13:21:39.350Z,bank
TXN006,5599.75,2026-07-06,BNKREF1006,settled,ICICI,2026-08-13T13:21:39.350Z,bank
TXN007,1889.25,2026-07-06,BNKREF1007,settled,SBIN,2026-08-13T13:21:39.350Z,bank
TXN008,999.0,2026-07-06,BNKREF1008,settled,HDFC,2026-08-13T13:21:39.350Z,bank
TXN010,15990.0,2026-07-06,BNKREF1010,settled,ICICI,2026-08-13T13:21:39.350Z,bank
TXN011,899.5,2026-07-06,BNKREF1011,settled,AXIS,2026-08-13T13:21:39.350Z,bank
TXN012,2248.0,2026-07-06,BNKREF1012,settled,ICICI,2026-08-13T13:21:39.350Z,bank


## Data quality validation

In [0]:
dq_transactions = (
    transactions_bronze
    .withColumn(
        "dq_negative_amount",
        F.col("amount") < 0
    )
    .withColumn(
        "dq_missing_txn_id",
        F.col("txn_id").isNull()
    )
    .withColumn(
        "dq_missing_channel",
        F.col("channel").isNull()
    )
)

display(
    dq_transactions.filter(
        F.col("dq_negative_amount") |
        F.col("dq_missing_txn_id") |
        F.col("dq_missing_channel")
    )
)

txn_id,amount,timestamp,status,merchant_id,currency,channel,MESSAGE_ARRIVAL_TIMESTAMP,source_system,dq_negative_amount,dq_missing_txn_id,dq_missing_channel


## Duplicate detection

In [0]:
duplicate_transactions = (
    transactions_bronze
    .groupBy("txn_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_transactions)

txn_id,count


## Deduplicate UPI responses

In [0]:
upi_window = Window.partitionBy("txn_id").orderBy(
    F.col("timestamp").desc()
)

upi_latest = (
    upi_bronze
    .withColumn("rn", F.row_number().over(upi_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(upi_latest)

error_message,note,payee_vpa,payer_vpa,response_code,timestamp,txn_id,upi_ref,MESSAGE_ARRIVAL_TIMESTAMP,source_system
null,null,merch101@upi,user1@paytm,00,2026-07-05T03:45:24.000Z,TXN001,UPIREF100001,2026-08-13T13:21:40.972Z,upi_npci
null,"card transaction, UPI log not applicable",null,null,00,2026-07-05T03:48:47.000Z,TXN002,null,2026-08-13T13:21:40.972Z,upi_npci
Insufficient funds,null,merch101@upi,user2@phonepe,05,2026-07-05T03:52:12.000Z,TXN003,null,2026-08-13T13:21:40.972Z,upi_npci
null,null,merch104@upi,user3@gpay,00,2026-07-05T04:43:01.000Z,TXN005,UPIREF100005,2026-08-13T13:21:40.972Z,upi_npci
null,null,merch105@upi,user4@paytm,00,2026-07-05T05:32:46.000Z,TXN007,UPIREF100007,2026-08-13T13:21:40.972Z,upi_npci
Timeout at issuer,null,merch101@upi,user5@phonepe,99,2026-07-05T05:45:09.000Z,TXN008,null,2026-08-13T13:21:40.972Z,upi_npci
Insufficient funds,null,merch103@upi,user6@gpay,05,2026-07-05T06:00:57.000Z,TXN009,null,2026-08-13T13:21:40.972Z,upi_npci
null,null,merch104@upi,user7@paytm,00,2026-07-05T06:48:35.000Z,TXN011,UPIREF100011,2026-08-13T13:21:40.972Z,upi_npci
null,null,merch107@upi,user8@phonepe,00,2026-07-05T07:32:50.000Z,TXN013,UPIREF100013,2026-08-13T13:21:40.972Z,upi_npci
null,null,merch101@upi,user9@gpay,00,2026-07-05T08:25:44.000Z,TXN015,UPIREF100015,2026-08-13T13:21:40.972Z,upi_npci


## Prepare transaction data

In [0]:
transactions_silver = (
    transactions_bronze
    .select(
        "txn_id",
        F.col("amount").alias("transaction_amount"),
        "channel",
        "MESSAGE_ARRIVAL_TIMESTAMP"
    )
)

display(transactions_silver)

txn_id,transaction_amount,channel,MESSAGE_ARRIVAL_TIMESTAMP
TXN001,1250.0,upi,2026-08-13T13:21:37.233Z
TXN002,3499.5,card,2026-08-13T13:21:37.233Z
TXN003,750.0,upi,2026-08-13T13:21:37.233Z
TXN004,12000.0,imps,2026-08-13T13:21:37.233Z
TXN005,299.0,upi,2026-08-13T13:21:37.233Z
TXN006,5600.75,card,2026-08-13T13:21:37.233Z
TXN007,1890.25,upi,2026-08-13T13:21:37.233Z
TXN008,999.0,upi,2026-08-13T13:21:37.233Z
TXN009,4500.0,imps,2026-08-13T13:21:37.233Z
TXN010,15999.0,card,2026-08-13T13:21:37.233Z


## Prepare settlement data

In [0]:
settlements_silver = (
    settlements_bronze
    .select(
        "txn_id",
        "settled_amount",
        "settlement_date",
        "bank_ref",
        "MESSAGE_ARRIVAL_TIMESTAMP"
    )
)

display(settlements_silver)

txn_id,settled_amount,settlement_date,bank_ref,MESSAGE_ARRIVAL_TIMESTAMP
TXN001,1249.0,2026-07-06,BNKREF1001,2026-08-13T13:21:39.350Z
TXN002,3498.5,2026-07-06,BNKREF1002,2026-08-13T13:21:39.350Z
TXN004,11995.0,2026-07-06,BNKREF1004,2026-08-13T13:21:39.350Z
TXN005,299.0,2026-07-06,BNKREF1005,2026-08-13T13:21:39.350Z
TXN006,5599.75,2026-07-06,BNKREF1006,2026-08-13T13:21:39.350Z
TXN007,1889.25,2026-07-06,BNKREF1007,2026-08-13T13:21:39.350Z
TXN008,999.0,2026-07-06,BNKREF1008,2026-08-13T13:21:39.350Z
TXN010,15990.0,2026-07-06,BNKREF1010,2026-08-13T13:21:39.350Z
TXN011,899.5,2026-07-06,BNKREF1011,2026-08-13T13:21:39.350Z
TXN012,2248.0,2026-07-06,BNKREF1012,2026-08-13T13:21:39.350Z


## Reconciliation

In [0]:
reconciled_df = (
    transactions_silver.alias("t")
    .join(
        settlements_silver.alias("s"),
        F.col("t.txn_id") == F.col("s.txn_id"),
        "left"
    )
    .join(
        upi_latest.alias("u"),
        F.col("t.txn_id") == F.col("u.txn_id"),
        "left"
    )
    .select(
        F.col("t.txn_id").alias("txn_id"),
        F.col("t.transaction_amount"),
        F.col("s.settled_amount"),
        (
            F.col("t.transaction_amount") -
            F.col("s.settled_amount")
        ).alias("difference"),
        F.when(
            F.col("s.txn_id").isNull(),
            F.lit("missing_in_settlement")
        )
        .when(
            F.abs(
                F.col("t.transaction_amount") -
                F.col("s.settled_amount")
            ) > 1,
            F.lit("mismatch")
        )
        .otherwise(
            F.lit("matched")
        )
        .alias("reconciliation_status"),
        F.col("u.response_code").alias("upi_response_code"),
        F.col("u.error_message").alias("upi_error_message"),
        F.col("t.channel"),
        F.col("s.settlement_date"),
        F.current_timestamp().alias("MESSAGE_ARRIVAL_TIMESTAMP")
    )
)

display(reconciled_df)

txn_id,transaction_amount,settled_amount,difference,reconciliation_status,upi_response_code,upi_error_message,channel,settlement_date,MESSAGE_ARRIVAL_TIMESTAMP
TXN001,1250.0,1249.0,1.0,matched,00,null,upi,2026-07-06,2026-08-13T13:23:36.623Z
TXN002,3499.5,3498.5,1.0,matched,00,null,card,2026-07-06,2026-08-13T13:23:36.623Z
TXN003,750.0,null,null,missing_in_settlement,05,Insufficient funds,upi,null,2026-08-13T13:23:36.623Z
TXN004,12000.0,11995.0,5.0,mismatch,null,null,imps,2026-07-06,2026-08-13T13:23:36.623Z
TXN005,299.0,299.0,0.0,matched,00,null,upi,2026-07-06,2026-08-13T13:23:36.623Z
TXN006,5600.75,5599.75,1.0,matched,null,null,card,2026-07-06,2026-08-13T13:23:36.623Z
TXN007,1890.25,1889.25,1.0,matched,00,null,upi,2026-07-06,2026-08-13T13:23:36.623Z
TXN008,999.0,999.0,0.0,matched,99,Timeout at issuer,upi,2026-07-06,2026-08-13T13:23:36.623Z
TXN009,4500.0,null,null,missing_in_settlement,05,Insufficient funds,imps,null,2026-08-13T13:23:36.623Z
TXN010,15999.0,15990.0,9.0,mismatch,null,null,card,2026-07-06,2026-08-13T13:23:36.623Z


## T+1 missing settlement logic

In [0]:
reconciled_df = (
    reconciled_df
    .withColumn(
        "t_plus_1_deadline",
        F.date_add(
            F.to_date("MESSAGE_ARRIVAL_TIMESTAMP"),
            1
        )
    )
    .withColumn(
        "is_t_plus_1_missing",
        (
            F.col("settled_amount").isNull()
            &
            (F.current_date() > F.col("t_plus_1_deadline"))
        )
    )
)

display(reconciled_df)

txn_id,transaction_amount,settled_amount,difference,reconciliation_status,upi_response_code,upi_error_message,channel,settlement_date,MESSAGE_ARRIVAL_TIMESTAMP,t_plus_1_deadline,is_t_plus_1_missing
TXN001,1250.0,1249.0,1.0,matched,00,null,upi,2026-07-06,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN002,3499.5,3498.5,1.0,matched,00,null,card,2026-07-06,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN003,750.0,null,null,missing_in_settlement,05,Insufficient funds,upi,null,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN004,12000.0,11995.0,5.0,mismatch,null,null,imps,2026-07-06,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN005,299.0,299.0,0.0,matched,00,null,upi,2026-07-06,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN006,5600.75,5599.75,1.0,matched,null,null,card,2026-07-06,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN007,1890.25,1889.25,1.0,matched,00,null,upi,2026-07-06,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN008,999.0,999.0,0.0,matched,99,Timeout at issuer,upi,2026-07-06,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN009,4500.0,null,null,missing_in_settlement,05,Insufficient funds,imps,null,2026-08-13T13:24:29.923Z,2026-08-14,false
TXN010,15999.0,15990.0,9.0,mismatch,null,null,card,2026-07-06,2026-08-13T13:24:29.923Z,2026-08-14,false


## Write Gold delta table

In [0]:
reconciled_df.write.format("delta").mode("overwrite").save(
    f"{BASE_PATH}/gold_reconciled_transactions"
)

## Create a SQL table

In [0]:
%sql
SELECT * FROM delta.`/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation/gold_reconciled_transactions`

txn_id,transaction_amount,settled_amount,difference,reconciliation_status,upi_response_code,upi_error_message,channel,settlement_date,MESSAGE_ARRIVAL_TIMESTAMP,t_plus_1_deadline,is_t_plus_1_missing
TXN001,1250.0,1249.0,1.0,matched,00,null,upi,2026-07-06,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN002,3499.5,3498.5,1.0,matched,00,null,card,2026-07-06,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN003,750.0,null,null,missing_in_settlement,05,Insufficient funds,upi,null,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN004,12000.0,11995.0,5.0,mismatch,null,null,imps,2026-07-06,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN005,299.0,299.0,0.0,matched,00,null,upi,2026-07-06,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN006,5600.75,5599.75,1.0,matched,null,null,card,2026-07-06,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN007,1890.25,1889.25,1.0,matched,00,null,upi,2026-07-06,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN008,999.0,999.0,0.0,matched,99,Timeout at issuer,upi,2026-07-06,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN009,4500.0,null,null,missing_in_settlement,05,Insufficient funds,imps,null,2026-08-13T13:24:46.192Z,2026-08-14,false
TXN010,15999.0,15990.0,9.0,mismatch,null,null,card,2026-07-06,2026-08-13T13:24:46.192Z,2026-08-14,false


## Match percentage

In [0]:
%sql
SELECT
    'Matched' AS status,
    SUM(
        CASE
            WHEN reconciliation_status = 'matched'
            THEN 1 ELSE 0
        END
    ) AS transaction_count
FROM delta.`/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation/gold_reconciled_transactions`

UNION ALL

SELECT
    'Unmatched' AS status,
    SUM(
        CASE
            WHEN reconciliation_status = 'matched'
            THEN 0 ELSE 1
        END
    ) AS transaction_count
FROM delta.`/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation/gold_reconciled_transactions`

status,transaction_count
Matched,8
Unmatched,7


Databricks visualization. Run in Databricks to view.

## Top UPI error codes

In [0]:
%sql
SELECT
    upi_response_code,
    COUNT(*) AS transaction_count
FROM delta.`/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation/gold_reconciled_transactions`
WHERE upi_response_code IS NOT NULL
GROUP BY upi_response_code
ORDER BY transaction_count DESC
LIMIT 5

upi_response_code,transaction_count
00,7
05,2
99,1


Databricks visualization. Run in Databricks to view.

## Daily volumne trend

In [0]:
%sql
SELECT
    DATE(MESSAGE_ARRIVAL_TIMESTAMP) AS transaction_date,
    COUNT(*) AS transaction_count,
    SUM(transaction_amount) AS total_amount
FROM delta.`/Workspace/Users/shashwatbangar@gmail.com/Drafts/FileStore/payments_reconciliation/gold_reconciled_transactions`
GROUP BY DATE(MESSAGE_ARRIVAL_TIMESTAMP)
ORDER BY transaction_date

transaction_date,transaction_count,total_amount
2026-08-13,15,54912.25


Databricks visualization. Run in Databricks to view.

## Data Quality dataframe

In [0]:
dq_df = (
    reconciled_df

    .withColumn(
        "dq_status",

        F.when(
            F.col("transaction_amount").isNull(),
            "MISSING_INTERNAL_AMOUNT"
        )

        .when(
            F.col("transaction_amount") < 0,
            "NEGATIVE_AMOUNT"
        )

        .when(
            F.col("difference").isNotNull() &
            (F.abs(F.col("difference")) > 1),
            "AMOUNT_MISMATCH"
        )

        .when(
            F.col("settled_amount").isNull(),
            "MISSING_SETTLEMENT"
        )

        .otherwise("PASS")
    )
)

display(dq_df)

txn_id,transaction_amount,settled_amount,difference,reconciliation_status,upi_response_code,upi_error_message,channel,settlement_date,MESSAGE_ARRIVAL_TIMESTAMP,t_plus_1_deadline,is_t_plus_1_missing,dq_status
TXN001,1250.0,1249.0,1.0,matched,00,null,upi,2026-07-06,2026-08-13T13:42:19.592Z,2026-08-14,false,PASS
TXN002,3499.5,3498.5,1.0,matched,00,null,card,2026-07-06,2026-08-13T13:42:19.592Z,2026-08-14,false,PASS
TXN003,750.0,null,null,missing_in_settlement,05,Insufficient funds,upi,null,2026-08-13T13:42:19.592Z,2026-08-14,false,MISSING_SETTLEMENT
TXN004,12000.0,11995.0,5.0,mismatch,null,null,imps,2026-07-06,2026-08-13T13:42:19.592Z,2026-08-14,false,AMOUNT_MISMATCH
TXN005,299.0,299.0,0.0,matched,00,null,upi,2026-07-06,2026-08-13T13:42:19.592Z,2026-08-14,false,PASS
TXN006,5600.75,5599.75,1.0,matched,null,null,card,2026-07-06,2026-08-13T13:42:19.592Z,2026-08-14,false,PASS
TXN007,1890.25,1889.25,1.0,matched,00,null,upi,2026-07-06,2026-08-13T13:42:19.592Z,2026-08-14,false,PASS
TXN008,999.0,999.0,0.0,matched,99,Timeout at issuer,upi,2026-07-06,2026-08-13T13:42:19.592Z,2026-08-14,false,PASS
TXN009,4500.0,null,null,missing_in_settlement,05,Insufficient funds,imps,null,2026-08-13T13:42:19.592Z,2026-08-14,false,MISSING_SETTLEMENT
TXN010,15999.0,15990.0,9.0,mismatch,null,null,card,2026-07-06,2026-08-13T13:42:19.592Z,2026-08-14,false,AMOUNT_MISMATCH


## Write Data Quality dataframe

In [0]:
dq_df.write.format("delta").mode("overwrite").save(
    f"{BASE_PATH}/dq_reconciliation"
)